In [7]:
import torch
from power_spherical import PowerSpherical

loc = torch.tensor([1.0, 0.0, 0.0])
scale = torch.tensor([1000.0])
dist = PowerSpherical(loc, scale)

In [8]:
import numpy as np
import torch
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display
from power_spherical import PowerSpherical

pio.renderers.default = "notebook"

# Wireframe sphere
u = np.linspace(0, 2 * np.pi, 40)
v = np.linspace(0, np.pi, 20)
sphere_traces = []
for vi in v:
    sphere_traces.append(
        go.Scatter3d(
            x=np.cos(u) * np.sin(vi),
            y=np.sin(u) * np.sin(vi),
            z=np.full_like(u, np.cos(vi)),
            mode="lines",
            line=dict(color="lightgray", width=1),
            showlegend=False,
            hoverinfo="skip",
        )
    )
for ui in u:
    sphere_traces.append(
        go.Scatter3d(
            x=np.cos(ui) * np.sin(v),
            y=np.sin(ui) * np.sin(v),
            z=np.cos(v),
            mode="lines",
            line=dict(color="lightgray", width=1),
            showlegend=False,
            hoverinfo="skip",
        )
    )

# Mean direction arrow
m = loc.numpy()
arrow_trace = go.Scatter3d(
    x=[0, m[0]],
    y=[0, m[1]],
    z=[0, m[2]],
    mode="lines+markers",
    line=dict(color="red", width=4),
    marker=dict(size=[0, 6], color="red"),
    name="Mean direction",
)


def sample_pts(scale_val):
    d = PowerSpherical(loc, torch.tensor([float(scale_val)]))
    return d.sample((10000,)).squeeze().detach().numpy()


pts0 = sample_pts(1)
samples_trace = go.Scatter3d(
    x=pts0[:, 0],
    y=pts0[:, 1],
    z=pts0[:, 2],
    mode="markers",
    marker=dict(size=2, color="steelblue", opacity=0.5),
    name="Samples",
)

no_axis = dict(
    showticklabels=False,
    showgrid=False,
    zeroline=False,
    showline=False,
    title="",
    showaxeslabels=False,
    visible=False,
)

fig = go.FigureWidget(data=sphere_traces + [samples_trace, arrow_trace])
fig.update_layout(
    width=600,
    height=600,
    scene=dict(xaxis=no_axis, yaxis=no_axis, zaxis=no_axis, aspectmode="cube", bgcolor="white"),
    margin=dict(l=0, r=0, b=0, t=10),
)

samples_idx = len(sphere_traces)

slider = widgets.FloatLogSlider(
    value=1.0,
    base=10,
    min=0,
    max=6,
    step=0.02,
    description="scale",
    continuous_update=True,
    layout=widgets.Layout(width="600px"),
)


def on_scale_change(change):
    pts = sample_pts(change["new"])
    with fig.batch_update():
        fig.data[samples_idx].x = pts[:, 0]
        fig.data[samples_idx].y = pts[:, 1]
        fig.data[samples_idx].z = pts[:, 2]


slider.observe(on_scale_change, names="value")

display(widgets.VBox([fig, slider]))

    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'lightgray', …